# A2 cross-task re-ID — train probe on execution, test on imagery

Same victim (motor-imagery decoder) and same chance baseline as A1, but the re-ID probe is trained on EEG of REAL movement (motor-execution runs) and asked to identify subjects from EEG of IMAGINED movement (motor-imagery runs). Tests whether identity rides on cognitive-task-orthogonal features.

Send back `results/04_a2_cross_task.json` and `runs/<run_id>/meta.json`. Figure regenerates locally.

**Runtime → L4 GPU.** Expected wall ~25 min.

**Don't `Save a copy in GitHub` from Colab** — overwrites the source notebook with cell-id and output-cell churn.

## 1. Clone repo (always pulls latest `main`)

In [ ]:
!rm -rf /content/bci-identity-leakage
!git clone https://github.com/manrajmondair/bci-identity-leakage.git /content/bci-identity-leakage
%cd /content/bci-identity-leakage
!git rev-parse HEAD

## 2. Confirm GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 3. Install project deps

In [ ]:
import torch
tv = torch.__version__.split('+')[0]
!pip install --quiet "torchaudio=={tv}"
!pip install --quiet mne moabb pyriemann braindecode opacus httpx tqdm rich

## 4. Prefetch PhysioNet imagery + execution runs (~3 min on archive mirror)

A2 needs both run families: imagery (4, 6, 8, 10, 12, 14) for the victim/test, execution (3, 5, 7, 9, 11, 13) for the probe-training data.

In [ ]:
!python -m data.prefetch_direct --runs imagery execution --workers 8

## 5. Run A2 cross-task re-ID — all three victim families

Expected wall: ~25 min.

In [ ]:
!PYTHONUNBUFFERED=1 python -u -m experiments.04_a2_cross_task --all

## 6. Emit run metadata + result JSON

In [ ]:
import json, os, subprocess, datetime, platform
import torch
git_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip()
now_utc = datetime.datetime.utcnow().replace(microsecond=0).isoformat() + 'Z'
run_id = now_utc.replace(':','').replace('-','').rstrip('Z') + f'_a2_cross_task_{git_sha[:7]}'
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
meta = {
    'run_id': run_id,
    'experiment': 'experiments.04_a2_cross_task',
    'args': ['--all'],
    'git_sha': git_sha,
    'hardware': f'Colab {gpu}',
    'platform': platform.platform(),
    'torch_version': torch.__version__,
    'completed_at_utc': now_utc,
    'outputs': ['results/04_a2_cross_task.json', 'figures/04_a2_cross_task.pdf'],
}
os.makedirs(f'runs/{run_id}', exist_ok=True)
with open(f'runs/{run_id}/meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('=== run metadata ===')
print(json.dumps(meta, indent=2))
print()
print('=== results/04_a2_cross_task.json ===')
with open('results/04_a2_cross_task.json') as f:
    print(f.read())

## 7. Download artifacts (drag both into the chat)

In [ ]:
from google.colab import files
files.download('results/04_a2_cross_task.json')
files.download(f'runs/{run_id}/meta.json')